# 1. Import Library 

In [23]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os
import pymysql
import sqlite3
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# 2. Driver set

In [25]:
options = webdriver.ChromeOptions()
options.add_argument("--headless")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# 3. url set 

In [26]:
# Webull 페이지 로드
url = 'https://www.webull.com/quote/us/gainers/1m'
driver.get(url)
time.sleep(5)  # JS 로딩 대기

# 4. Get Data

## copy selector

In [ ]:
# name
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(2) > div > div.detail.canClick > p.tit.bold

# last price
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(5) > span

# change
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(4) > div > span

# high & low
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(6)
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(7)

# Volume
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(8) > div > span

# Market cap
#app > div.csr78 > div.csr137 > div.csr141.csr145.csr127 > div.table-body > div:nth-child(1) > div:nth-child(11) > div > span

## data check

In [ ]:
# page source
html = driver.page_source
soup = BeautifulSoup(html, 'html.parser')

# Company name
company_tags = soup.select("p.tit.bold")

for tag in company_tags:
    print(tag.text.strip())

driver.quit()


Abivax S.A.
Bitmine Immersion Technologies Inc
Trident Digital Tech Holdings Ltd
CEA Industries
Incannex Healthcare Ltd
Lixte Biotechnology Hldgs Inc
Zepp Health Corporation
Connexa Sports Technologies Inc
Mega Matrix Corp
ProKidney
Aeye Inc
Opendoor Technologies Inc
Wolfspeed Inc
Zooz Power Ltd
Sonnet Biotherapeutc Hldng Inc
CEL-SCI Corp
180 Life Sciences Corp
Dallasnews Corporation
Elong Power Holding Limited
Mill City Ventur
Newegg Commerce Inc
Currenc Group Inc
Aureus Greenway Holdings Inc.
SMART DIGITAL GROUP LTD
Cyclacel Pharma
Neumora Therapeutics Inc
Bit Origin Limited
Stem Inc
Celcuity Inc
MoneyHero Limited
Mink Therapeutics, Inc.
CALCIMEDICA INC
Scilex Holding Co
Healthcare Triangle Inc
Calidi Biotherapeutics Inc
Viomi Technology Co., Ltd.
Raytech Holding Limited
Cambium Networks
Safety Shot
Btcs Inc
COGNITION THERAPEUTICS INC
Stardust Power Inc
Nvni Group Limited
DRAGONFLY ENERGY HOLDINGS CORP
Vor Biopharma Inc.
Planet Green
Upexi Inc
Future Fintech G
Burning Rock Biotech Lt

In [38]:
# Last Price
last_price = soup.select("#app .table-body div:nth-child(5) > span")

for p in last_price:
    print(p.text.strip())

driver.quit()

66.57
35.11
1.450
57.59
1.190
4.070
13.73
4.300
3.780
3.080
4.050
2.340
1.680
3.510
4.050
8.61
2.910
14.81
3.250
5.94
39.08
2.160
1.880
25.23
15.39
2.280
0.4723
18.91
36.79
2.040
18.71
3.710
15.52
0.0579
0.5496
3.200
2.540
0.9201
0.7232
5.16
0.7151
0.4606
0.6881
0.3701
2.420
2.050
5.88
2.550
6.90
20.93


## data preprocessing

In [ ]:
# data preprocessing
def parse(text):
    if text.endswith("B"):
        return float(text[:-1]) * 1_000_000_000
    elif text.endswith("M"):
        return float(text[:-1]) * 1_000_000
    elif text.endswith("K"):
        return float(text[:-1]) * 1_000
    else:
        return float(text)

In [56]:
# 종목 단위 추출 (행 단위)
rows = soup.select("div.table-body > div.table-row")

for row in rows:
    try:
        name = row.select_one("p.tit.bold").text.strip()
        price = row.select_one("div:nth-child(5) > span").text.strip()
        change = parse(row.select_one("div:nth-child(4) > div > span").text.strip().replace("%", "").replace("+", ""))
        high = row.select_one("div:nth-child(6)").text.strip()
        low = row.select_one("div:nth-child(7)").text.strip()
        volume = parse(row.select_one("div:nth-child(8) > div > span").text.strip())
        market_cap = parse(row.select_one("div:nth-child(11) > div > span").text.strip())
        print(f"회사: {name}, 현재가: {price}, 변동률:{change}, 고가: {high}, 저가: {low}, 거래량: {volume}, 시가총액: {market_cap}")
    
    except AttributeError:
        continue  # 요소가 없는 경우는 건너뜀

회사: Abivax S.A., 현재가: 66.57, 변동률:799.59, 고가: 68.12, 저가: 65.67, 거래량: 1720000.0, 시가총액: 4219999999.9999995
회사: Bitmine Immersion Technologies Inc, 현재가: 35.11, 변동률:723.21, 고가: 41.49, 저가: 34.39, 거래량: 48780000.0, 시가총액: 3940000000.0
회사: Trident Digital Tech Holdings Ltd, 현재가: 1.450, 변동률:574.42, 고가: 1.480, 저가: 1.260, 거래량: 2020000.0, 시가총액: 111530000.0
회사: CEA Industries, 현재가: 57.59, 변동률:4.0, 고가: 82.88, 저가: 46.10, 거래량: 14360000.0, 시가총액: 48450000.0
회사: Incannex Healthcare Ltd, 현재가: 1.190, 변동률:485.34, 고가: 1.250, 저가: 1.100, 거래량: 79660000.0, 시가총액: 35030000.0
회사: Lixte Biotechnology Hldgs Inc, 현재가: 4.070, 변동률:469.55, 고가: 4.270, 저가: 4.000, 거래량: 83310.0, 시가총액: 21770000.0
회사: Zepp Health Corporation, 현재가: 13.73, 변동률:438.43, 고가: 13.79, 저가: 11.16, 거래량: 476870.0, 시가총액: 197280000.0
회사: Connexa Sports Technologies Inc, 현재가: 4.300, 변동률:378.42, 고가: 4.380, 저가: 3.150, 거래량: 8.0, 시가총액: 62620000.0
회사: Mega Matrix Corp, 현재가: 3.780, 변동률:372.5, 고가: 3.900, 저가: 2.700, 거래량: 4780000.0, 시가총액: 154260000.0
회사: ProKidney, 현재가

# 5. Save data 

## SQLite & CSV

In [58]:
import csv
# CSV 저장 준비
csv_filename = "webull_stocks.csv"
csv_fields = ["name", "price", "change", "high", "low", "volume", "market_cap"]

with open(csv_filename, "w", newline="", encoding="utf-8-sig") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_fields)
    writer.writeheader()

    # SQLite 저장 준비
    conn = sqlite3.connect("webull_stocks.db")
    cursor = conn.cursor()

    # 테이블 생성
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS stocks (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT,
            price REAL,
            change REAL,
            high REAL,
            low REAL,
            volume REAL,
            market_cap REAL
        )
    """)

    # 데이터 추출
    rows = soup.select("div.table-body > div.table-row")

    for row in rows:
        try:
            name = row.select_one("p.tit.bold").text.strip()
            price = float(row.select_one("div:nth-child(5) > span").text.strip())
            change = parse(row.select_one("div:nth-child(4) > div > span").text.strip().replace("%", "").replace("+", ""))
            high = float(row.select_one("div:nth-child(6)").text.strip())
            low = float(row.select_one("div:nth-child(7)").text.strip())
            volume = parse(row.select_one("div:nth-child(8) > div > span").text.strip())
            market_cap = parse(row.select_one("div:nth-child(11) > div > span").text.strip())
            print(f"회사: {name}, 현재가: {price}, 변동률:{change}, 고가: {high}, 저가: {low}, 거래량: {volume}, 시가총액: {market_cap}")

            # CSV 저장
            writer.writerow({
                "name": name,
                "price": price,
                "change": change,
                "high": high,
                "low": low,
                "volume": volume,
                "market_cap": market_cap
            })

            # SQL 저장
            cursor.execute("""
                INSERT INTO stocks (name, price, change, high, low, volume, market_cap)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (name, price, change, high, low, volume, market_cap))

            print(f"저장 완료: {name}")

        except AttributeError:
            continue

    conn.commit()
    conn.close()

print(f"\nCSV 저장 완료: {csv_filename}")
print("SQLite DB 저장 완료: webull_stocks.db")


회사: Abivax S.A., 현재가: 66.57, 변동률:799.59, 고가: 68.12, 저가: 65.67, 거래량: 1720000.0, 시가총액: 4219999999.9999995
저장 완료: Abivax S.A.
회사: Bitmine Immersion Technologies Inc, 현재가: 35.11, 변동률:723.21, 고가: 41.49, 저가: 34.39, 거래량: 48780000.0, 시가총액: 3940000000.0
저장 완료: Bitmine Immersion Technologies Inc
회사: Trident Digital Tech Holdings Ltd, 현재가: 1.45, 변동률:574.42, 고가: 1.48, 저가: 1.26, 거래량: 2020000.0, 시가총액: 111530000.0
저장 완료: Trident Digital Tech Holdings Ltd
회사: CEA Industries, 현재가: 57.59, 변동률:4.0, 고가: 82.88, 저가: 46.1, 거래량: 14360000.0, 시가총액: 48450000.0
저장 완료: CEA Industries
회사: Incannex Healthcare Ltd, 현재가: 1.19, 변동률:485.34, 고가: 1.25, 저가: 1.1, 거래량: 79660000.0, 시가총액: 35030000.0
저장 완료: Incannex Healthcare Ltd
회사: Lixte Biotechnology Hldgs Inc, 현재가: 4.07, 변동률:469.55, 고가: 4.27, 저가: 4.0, 거래량: 83310.0, 시가총액: 21770000.0
저장 완료: Lixte Biotechnology Hldgs Inc
회사: Zepp Health Corporation, 현재가: 13.73, 변동률:438.43, 고가: 13.79, 저가: 11.16, 거래량: 476870.0, 시가총액: 197280000.0
저장 완료: Zepp Health Corporation
회사: Connexa Sports

## pymySQL

In [61]:
import pymysql


con = pymysql.connect(
    host='localhost',
    user='root',
    password='1234',
    db='Webull_trade',
    charset='utf8mb4',
)
cursor = con.cursor()

create_db_query = "CREATE DATABASE IF NOT EXISTS Webull_trade CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci;"
cursor.execute(create_db_query)


cursor.execute('''
CREATE TABLE IF NOT EXISTS stocks (
    id INT AUTO_INCREMENT PRIMARY KEY,
    company_name VARCHAR(255),
    current_price FLOAT,
    change_percent FLOAT,
    high_price FLOAT,
    low_price FLOAT,
    trading_volume FLOAT,
    market_cap FLOAT
)
''')


insert_sql = """
    INSERT INTO stocks (
        company_name, current_price, change_percent,
        high_price, low_price, trading_volume, market_cap
    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
"""


for row in rows:
    try:
        company_name = row.select_one("p.tit.bold").text.strip()
        current_price = float(row.select_one("div:nth-child(5) > span").text.strip().replace(",", ""))
        change_percent = float(row.select_one("div:nth-child(4) > div > span").text.strip().replace("%", "").replace("+", ""))
        high_price = float(row.select_one("div:nth-child(6)").text.strip().replace(",", ""))
        low_price = float(row.select_one("div:nth-child(7)").text.strip().replace(",", ""))
        trading_volume = float(parse(row.select_one("div:nth-child(8) > div > span").text.strip()))
        market_cap = float(parse(row.select_one("div:nth-child(11) > div > span").text.strip()))

        cursor.execute(insert_sql, (
            company_name, current_price, change_percent,
            high_price, low_price, trading_volume, market_cap
        ))

        print(f"저장 완료: {company_name}")

    except AttributeError:
        continue
    except Exception as e:
        print(f"[에러 발생] {e}")
        continue

con.commit()
cursor.close()
con.close()
print("MySQL 저장 완료 ")


저장 완료: Abivax S.A.
저장 완료: Bitmine Immersion Technologies Inc
저장 완료: Trident Digital Tech Holdings Ltd
저장 완료: CEA Industries
저장 완료: Incannex Healthcare Ltd
저장 완료: Lixte Biotechnology Hldgs Inc
저장 완료: Zepp Health Corporation
저장 완료: Connexa Sports Technologies Inc
저장 완료: Mega Matrix Corp
저장 완료: ProKidney
저장 완료: Aeye Inc
저장 완료: Opendoor Technologies Inc
저장 완료: Wolfspeed Inc
저장 완료: Zooz Power Ltd
저장 완료: Sonnet Biotherapeutc Hldng Inc
저장 완료: CEL-SCI Corp
저장 완료: 180 Life Sciences Corp
저장 완료: Dallasnews Corporation
저장 완료: Elong Power Holding Limited
저장 완료: Mill City Ventur
저장 완료: Newegg Commerce Inc
저장 완료: Currenc Group Inc
저장 완료: Aureus Greenway Holdings Inc.
저장 완료: SMART DIGITAL GROUP LTD
저장 완료: Cyclacel Pharma
저장 완료: Neumora Therapeutics Inc
저장 완료: Bit Origin Limited
저장 완료: Stem Inc
저장 완료: Celcuity Inc
저장 완료: MoneyHero Limited
저장 완료: Mink Therapeutics, Inc.
저장 완료: CALCIMEDICA INC
저장 완료: Scilex Holding Co
저장 완료: Healthcare Triangle Inc
저장 완료: Calidi Biotherapeutics Inc
저장 완료: Viomi Technolog